[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Decorators


## What you will be able to do

Read a line beginning with `@` and say exactly what it does, and write decorators of your own:
ones that wrap a function, ones that take settings, and ones that stack, including on methods.


## The idea

### The problem

**Properties**, **Class and Static Methods** and **Dataclasses** each put a single line above a
method or a class, beginning with `@`, and the method or class then behaves differently. In the
**Testing and Packaging** guide, `@pytest.fixture` does the same to a function. It looks like
special syntax, a keyword that switches something on. It is not.

Here is a problem it solves. Three functions in a program read from an instrument over a bad
connection. Each should log its arguments and its result, and each should try again, up to three
times, when the connection drops. Written into each function, the logging and retrying make every
function ten lines long, when one line does the reading, and they are copied three times. Change
the log format and there are three copies to edit, and one of them will be missed.

What you want is to write the logging once and the retrying once, and attach either one to any
function with a single line.

### What a decorator is

> A **decorator** is a function that takes a function and returns a function. Writing
> `@decorator` on the line above `def name` is shorthand for `name = decorator(name)`, run once,
> when the `def` runs. Whatever the decorator returns takes over the original's name.

### Why it works that way

Nothing here is new. Three facts from the **Python from the Start** guide do all of the work:

- A function is a value, so it can be passed in and handed back, as the **Functions** notebook
  showed.
- `*args` and `**kwargs` let one function accept whatever another accepts, also from
  **Functions**.
- A function can be defined inside another, and the inner one remembers the variables around it.
  That is a closure, from the **Scope** notebook.

A decorator is those three put together. The outer function receives the original. Inside it, a new
function, conventionally called `wrapper`, accepts any arguments, does something first, calls the
original through the closure, does something after, and returns the result. The outer function
returns `wrapper`, and `@` puts it under the original's name.

Four consequences follow, and each has a section below. Decoration happens once, when the function
is defined, not on every call. The wrapper replaces the original's name and docstring unless they
are copied across, which `functools.wraps` does. A decorator that takes its own settings, such as
`@retry(times=3)`, needs one more layer, because `retry(times=3)` runs first and has to return the
decorator. And stacked decorators apply from the bottom up.

### Where you will meet this

Mostly in code you use rather than write: `@property` in **Properties**, `@classmethod` and
`@staticmethod` in **Class and Static Methods**, `@dataclass` in **Dataclasses**, and
`@pytest.fixture` in the **Testing and Packaging** guide. Knowing the mechanism matters most when
something goes wrong, because the traceback will run through functions you never wrote.

### What this notebook covers

The same small program written twice, first without decorators and then with them, so the payoff
is visible before any of the mechanics. Then the equivalence between `@` and assignment, a first
wrapper, keeping its name with
`functools.wraps`, decorators that take settings, stacking, decorating a method, decorators that
register a function rather than wrap it, and two from the standard library, including the shorter
way to write a context manager. Then one example that puts them together.

### A first look

A decorator, and one function it changes. There is nothing to run yet: read it, and read the output
underneath it.

```python
def shout(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper


@shout
def label(name):
    return f"station {name}"


print(label("Tromso"))
```

```
STATION TROMSO
```

`label` was written to return `station Tromso`. The `@shout` line is the only reason it did not.


## Setup

Two imports, and a stand-in for a flaky connection.

- `functools` provides `wraps`, which keeps a decorated function's name and docstring, and
  `cache`, one of the standard library's own decorators
- `contextlib` provides `contextmanager`, the shorter way to write the context managers from the
  **Context Managers and Iterators** notebook

`Connection` stands in for a link to an instrument over a bad line. It raises `ConnectionError`
for its first few requests, as many as `drops` says, and then answers from `READINGS`, so every
run of this notebook fails in exactly the same places. Its `__repr__` lets a log line show which
connection it was given. The comparison at the start of the worked examples uses it.

Every decorator in this notebook is written in the section that explains it.

**Run this cell before the rest of the notebook.**


In [1]:
import functools
import contextlib

READINGS = {"temperature": -4.1, "pressure": 1013.2, "humidity": 81}


class Connection:
    """A link to an instrument that drops its first few requests."""

    def __init__(self, name, drops):
        self.name = name
        self.drops = drops

    def __repr__(self):
        return f"Connection({self.name!r})"

    def request(self, quantity):
        if self.drops > 0:
            self.drops -= 1
            raise ConnectionError("no response")
        return READINGS[quantity]


print("ready")


ready


## Worked examples

### Before and after: one program, written twice

Here is the problem from the top of this notebook, in code. Three functions read from an instrument
over a connection that drops requests. Each should log the call and its result, and try again, up to
three times, when the connection drops.

First, without decorators. Read one function closely, then compare the other two with it.


In [2]:
def read_temperature(link):
    print(f"  calling read_temperature({link!r})")
    for attempt in range(1, 4):
        try:
            result = link.request("temperature")
            print(f"  read_temperature returned {result!r}")
            return result
        except ConnectionError as error:
            print(f"  attempt {attempt} of 3 failed: {error}")
    raise ConnectionError("read_temperature failed 3 times")


def read_pressure(link):
    print(f"  calling read_pressure({link!r})")
    for attempt in range(1, 4):
        try:
            result = link.request("pressure")
            print(f"  read_pressure returned {result!r}")
            return result
        except ConnectionError as error:
            print(f"  attempt {attempt} of 3 failed: {error}")
    raise ConnectionError("read_pressure failed 3 times")


def read_humidity(link):
    print(f"  calling read_humidity({link!r})")
    for attempt in range(1, 4):
        try:
            result = link.request("humidity")
            print(f"  read_humidity returned {result!r}")
            return result
        except ConnectionError as error:
            print(f"  attempt {attempt} of 3 failed: {error}")
    raise ConnectionError("read_humidity failed 3 times")


Now run them. Each call gets a new connection that drops a set number of requests: two, then one,
then none.


In [3]:
temperature = read_temperature(Connection("north", drops=2))
pressure = read_pressure(Connection("north", drops=1))
humidity = read_humidity(Connection("north", drops=0))


  calling read_temperature(Connection('north'))
  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  read_temperature returned -4.1
  calling read_pressure(Connection('north'))
  attempt 1 of 3 failed: no response
  read_pressure returned 1013.2
  calling read_humidity(Connection('north'))
  read_humidity returned 81


It works. Here is precisely what is wrong with it.

- **The work is buried.** Each function is ten lines long, and the reading itself is one of them,
  `link.request(...)`. The rest is the function's first line and the logging and retrying around the
  reading.
- **The same logic is copied three times.** The three functions differ only in the quantity they ask
  for and in their own names. Everything else has to be kept identical by hand.
- **Each copy names itself.** `read_pressure` appears three times inside its own body, in strings.
  Rename the function, and its log goes on printing the old name unless all three are found.
- **A small change means many edits.** Five attempts instead of three means changing the `4` in
  `range` and the `3` in both messages, in all three functions: nine edits that must agree.
- **The next function costs another copy.** A fourth reading means pasting ten more lines and
  editing four of them.

Now the same program with decorators. The logging and the retrying are each written once.


In [4]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"  calling {func.__name__}({', '.join(shown)})")
        result = func(*args, **kwargs)
        print(f"  {func.__name__} returned {result!r}")
        return result
    return wrapper


def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    print(f"  attempt {attempt} of {times} failed: {error}")
            raise ConnectionError(f"{func.__name__} failed {times} times")
        return wrapper
    return decorator


How these two work is the rest of this notebook. For now, notice only that each one exists once.
Here are the three reading functions again, with the two decorators attached.


In [5]:
@log_calls
@retry(times=3)
def read_temperature(link):
    return link.request("temperature")


@log_calls
@retry(times=3)
def read_pressure(link):
    return link.request("pressure")


@log_calls
@retry(times=3)
def read_humidity(link):
    return link.request("humidity")


Each function is now four lines: two that attach the behavior, and two that are the function
itself. The body is nothing but the reading. Run them with the same connections as before.


In [6]:
temperature = read_temperature(Connection("north", drops=2))
pressure = read_pressure(Connection("north", drops=1))
humidity = read_humidity(Connection("north", drops=0))


  calling read_temperature(Connection('north'))
  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  read_temperature returned -4.1
  calling read_pressure(Connection('north'))
  attempt 1 of 3 failed: no response
  read_pressure returned 1013.2
  calling read_humidity(Connection('north'))
  read_humidity returned 81


Line for line, the same output as the version without decorators. The program does exactly what it
did before. What changed is where each part of it lives.

Now the change that took nine edits before. The humidity sensor has a worse connection, and needs
five attempts.


In [7]:
@log_calls
@retry(times=5)
def read_humidity(link):
    return link.request("humidity")


humidity = read_humidity(Connection("north", drops=4))

print()
try:
    read_pressure(Connection("north", drops=4))
except ConnectionError as error:
    print("  gave up:", error)


  calling read_humidity(Connection('north'))
  attempt 1 of 5 failed: no response
  attempt 2 of 5 failed: no response
  attempt 3 of 5 failed: no response
  attempt 4 of 5 failed: no response
  read_humidity returned 81

  calling read_pressure(Connection('north'))
  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  attempt 3 of 3 failed: no response
  gave up: read_pressure failed 3 times


One number changed, on the one function that needed it. Given the same bad connection,
`read_pressure` still gives up after three attempts, and neither decorator was edited.

| | Without decorators | With decorators |
|---|---|---|
| Lines in each reading function | 10 | 4 |
| Logging and retrying written | once per function | once in total |
| Five attempts instead of three | nine edits across three functions | `times=5` on the function that needs it |
| Adding a fourth reading | paste ten lines, edit four of them | four lines, two of them `@` lines |
| Renaming a function | fix three strings inside it | nothing, because the log reads the name itself |

That is what decorators are for: behavior that surrounds many functions, written once, and attached
with a single line.

The rest of this notebook takes the second version apart, one question at a time.

| Question about the second version | The section that answers it |
|---|---|
| What does the `@` line do? | `@` is reassignment, and it runs once |
| How does `log_calls` get in front of and behind every call? | Your first wrapper |
| Why does the log say `read_temperature`, not `wrapper`? | Keeping the name: `functools.wraps` |
| Why does `retry` take `times=3` in parentheses, and `log_calls` nothing? | A decorator that takes an argument |
| Why is `log_calls` above `retry`, not below it? | Stacking decorators, and the closing example |


### `@` is reassignment, and it runs once

A decorator that does nothing but print shows when it runs. `mark` takes a function, announces it,
and hands the same function back.


In [8]:
def mark(func):
    print(f"  mark is decorating {func.__name__}")
    return func


@mark
def average(values):
    return round(sum(values) / len(values), 2)


print("defined. Now two calls:")
print(average([-4.1, -2.6]), average([1.0, 3.0]))


  mark is decorating average
defined. Now two calls:
-3.35 2.0


`mark` printed once, while the `def` was running, and not at all during the two calls. Decoration
happens when the function is defined.

Here is the same thing without `@`.


In [9]:
def middle(values):
    ordered = sorted(values)
    return ordered[len(ordered) // 2]


middle = mark(middle)

print(middle([-4.1, -2.6, -3.8]))


  mark is decorating middle
-3.8


That is exactly what `@mark` did, written out: pass the function in, and put whatever comes back
under the same name. `@` adds no ability of its own. It saves writing the name three times, and it
puts the decoration where a reader sees it, at the top of the function rather than somewhere below.

### Your first wrapper

`mark` returned the function it was given, so calling it changed nothing. A decorator changes
behavior by returning a **different** function: one that does something, calls the original, and
does something else. By convention it is called `wrapper`.

`*args` and `**kwargs` are what let one `wrapper` serve any function. Whatever the caller passes,
`wrapper` receives and forwards unchanged, so it works for a function of one argument or of ten.

This is the `log_calls` from the comparison at the start, built up in steps. This first version
leaves out one line, `functools.wraps`, and the section after it shows what that line is for.


In [10]:
def log_calls(func):
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"  calling {func.__name__}({', '.join(shown)})")
        result = func(*args, **kwargs)
        print(f"  {func.__name__} returned {result!r}")
        return result
    return wrapper


@log_calls
def average(values, places=2):
    """Mean of a list of readings."""
    return round(sum(values) / len(values), places)


average([-4.1, -2.6, -3.8], places=1)


  calling average([-4.1, -2.6, -3.8], places=1)
  average returned -3.5


-3.5

Both log lines came from `wrapper`, and the value between them came from the original `average`,
which `wrapper` reached through the closure. `func` is a variable of `log_calls`, and `wrapper`
remembers it after `log_calls` has returned. That is the **Scope** notebook's closure doing the
real work.

The last line of `wrapper` matters most. `return result` hands the original's answer back to the
caller. Leave it out and every decorated function returns `None`, which is the quiet error at the
end of this notebook.

### What the wrapper did to the name

The name `average` now holds `wrapper`, and it answers to `wrapper`'s name.


In [11]:
print("average.__name__:", average.__name__)
print("average.__doc__: ", average.__doc__)


average.__name__: wrapper
average.__doc__:  None


`average` reports `wrapper`'s name and `wrapper`'s docstring, which is `None`. `help(average)` would
describe `wrapper` too. In a program with twenty decorated functions, every one of them now calls
itself `wrapper`.

### Keeping the name: `functools.wraps`

`functools.wraps` is itself a decorator, applied to `wrapper`. It copies the original's name,
docstring and a few other details onto `wrapper`, and stores the original as `__wrapped__`.


In [12]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        shown = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"  calling {func.__name__}({', '.join(shown)})")
        result = func(*args, **kwargs)
        print(f"  {func.__name__} returned {result!r}")
        return result
    return wrapper


@log_calls
def average(values, places=2):
    """Mean of a list of readings."""
    return round(sum(values) / len(values), places)


print("average.__name__:", average.__name__)
print("average.__doc__: ", average.__doc__)
print("the original, unlogged:", average.__wrapped__([-4.1, -2.6]))


average.__name__: average
average.__doc__:  Mean of a list of readings.
the original, unlogged: -3.35


The decorated `average` reports its own name and docstring again. `__wrapped__` reaches past the
wrapper to the original, which is useful in a test that wants to check the function without its
logging.

Use `functools.wraps` on every wrapper you write. It costs one line. The errors at the end of this
notebook include a case where it changes what an error message says, which is worth understanding
before it surprises you.

### A decorator that takes an argument

This is the `retry` from the comparison at the start, taken apart.

`@retry(times=3)` has parentheses, which means `retry(times=3)` is called **first**, and whatever it
returns is then used as the decorator. So `retry` is not a decorator. It is a function that builds
one, and there are three layers:

- `retry(times)` receives the setting, and returns `decorator`
- `decorator(func)` receives the function, and returns `wrapper`
- `wrapper(*args, **kwargs)` runs on every call

Each inner layer remembers the ones outside it, so `wrapper` can see both `times` and `func`.


In [13]:
def retry(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    print(f"  attempt {attempt} of {times} failed: {error}")
            raise ConnectionError(f"{func.__name__} failed {times} times")
        return wrapper
    return decorator


print("retry(3) returns a", type(retry(3)).__name__, "called", retry(3).__name__)


retry(3) returns a function called decorator


`retry(3)` is a function called `decorator`, not a decorated anything. The `@` line then hands the
function below it to that.

To make a connection fail in a fixed pattern, `outcomes` is an iterator, from the **Context Managers
and Iterators** notebook: two timeouts, then a reading.


In [14]:
outcomes = iter(["timeout", "timeout", -4.1])


@retry(times=3)
def read_sensor():
    outcome = next(outcomes)
    if outcome == "timeout":
        raise ConnectionError("no response")
    return outcome


print("got:", read_sensor())


  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
got: -4.1


The first two attempts raised `ConnectionError`, `wrapper` caught each one and tried again, and the
third returned a reading. The function body knows nothing about retrying.

When every attempt fails, the wrapper has to say so.


In [15]:
outcomes = iter(["timeout", "timeout", "timeout"])

try:
    read_sensor()
except ConnectionError as error:
    print("gave up:", error)


  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  attempt 3 of 3 failed: no response
gave up: read_sensor failed 3 times


After the last attempt, `wrapper` raises a `ConnectionError` of its own, naming the function, so the
caller still finds out. A retry that swallowed the final failure would turn a dead instrument into a
reading of `None`.

### Stacking decorators

More than one decorator can sit above a function. They apply from the bottom up: the one nearest the
`def` wraps the function first, and each line above wraps the result.


In [16]:
def tag(label):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return f"<{label}>{func(*args, **kwargs)}</{label}>"
        return wrapper
    return decorator


@tag("b")
@tag("i")
def station():
    return "Tromso"


print(station())


<b><i>Tromso</i></b>


`<i>` is on the inside because `tag("i")` was applied first. Written out, the two lines mean
`station = tag("b")(tag("i")(station))`.

Order is not cosmetic. The closing example applies the same two decorators both ways round and gets
different behavior.

### Decorating a method

A method is a function defined in a class, as the **Methods** notebook established, so decorating one
works the same way. `self` arrives in `wrapper` as the first of `*args` and is passed along with
everything else.


In [17]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    def __repr__(self):
        return f"Station({self.name!r})"

    @log_calls
    def average(self):
        return round(sum(self.readings) / len(self.readings), 2)


Station("Tromso", [-4.1, -2.6]).average()


  calling average(Station('Tromso'))
  average returned -3.35


-3.35

The log line shows the station as the first argument, because `self` is the first argument. It
prints readably only because `Station` has a `__repr__`, from the **Dunder Methods** notebook.
Without one, the log would show a memory address.

### Decorators that do not wrap

A decorator can hand back the function it was given, unchanged, after doing something with it. The
commonest use is registration: recording functions in a table so that something else can find them
later.


In [18]:
CHECKS = {}


def check(func):
    CHECKS[func.__name__] = func
    return func


@check
def not_empty(readings):
    return len(readings) > 0


@check
def in_range(readings):
    return all(-90 <= r <= 60 for r in readings)


for name, rule in CHECKS.items():
    print(f"  {name:<10} {rule([-4.1, 999.0])}")

print("not_empty is still a plain", type(not_empty).__name__, "named", not_empty.__name__)


  not_empty  True
  in_range   False
not_empty is still a plain function named not_empty


Neither check was wrapped. The decorator's only effect was the entry it made in `CHECKS`, and the loop
ran every registered rule without naming either one. Adding a third check means writing it with
`@check` above it; nothing else changes.

The same idea, marking a function so that something else can find it later, is how `@pytest.fixture`
works in the **Testing and Packaging** guide.

### Two from the standard library

`functools.cache` remembers what a function returned for each argument, and returns the remembered
value instead of calling the function again. It suits a function that is slow and always gives the
same answer for the same input.


In [19]:
@functools.cache
def elevation(name):
    print(f"  looking up {name}")
    return {"Tromso": 100, "Bodo": 11}[name]


print(elevation("Tromso"), elevation("Tromso"), elevation("Bodo"))
print(elevation.cache_info())


  looking up Tromso
  looking up Bodo
100 100 11
CacheInfo(hits=1, misses=2, maxsize=None, currsize=2)


`looking up Tromso` printed once for two calls, because the second call never reached the function
body. `cache_info` counts it: one hit, two misses.

The condition matters. A cached function that depends on anything besides its arguments, such as the
time or the contents of a file, keeps returning its first answer after the truth has changed.

`contextlib.contextmanager` is the shorter way to write a context manager. The decorated function is
a generator: the code before `yield` does the work of `__enter__`, the value it yields is what `as`
binds, and the code after it does the work of `__exit__`.


In [20]:
@contextlib.contextmanager
def section(title):
    print(f"  == {title} ==")
    try:
        yield title.upper()
    finally:
        print(f"  == end {title} ==")


with section("north") as heading:
    print("  as bound:", heading)

try:
    with section("south"):
        raise ValueError("instrument fault")
except ValueError as error:
    print("  caught:", error)


  == north ==
  as bound: NORTH
  == end north ==
  == south ==
  == end south ==
  caught: instrument fault


Four lines of generator replace a class with `__enter__` and `__exit__`, and the closing line printed
even when the block raised.

The `try` and `finally` are not optional. When the block raises, the exception is raised inside the
generator at the `yield`, so anything after the `yield` is skipped unless a `finally` protects it.
Here is the same context manager without them.


In [21]:
@contextlib.contextmanager
def section_unprotected(title):
    print(f"  == {title} ==")
    yield
    print(f"  == end {title} ==")


try:
    with section_unprotected("east"):
        raise ValueError("instrument fault")
except ValueError as error:
    print("  caught, and the closing line never printed:", error)


  == east ==
  caught, and the closing line never printed: instrument fault


The class version in **Context Managers and Iterators** ran `__exit__` on every way out
automatically. The generator version only cleans up on an error when you write `finally`, and that
is the price of the shorter form.

### The decorators coming up in this guide

| Decorator | What it does | Where |
|---|---|---|
| `@property` | makes a method read like an attribute | **Properties** |
| `@name.setter` | runs a method when that attribute is assigned | **Properties** |
| `@classmethod` | passes the class instead of an object | **Class and Static Methods** |
| `@staticmethod` | passes neither | **Class and Static Methods** |
| `@dataclass` | writes `__init__`, `__repr__` and `__eq__` for a class | **Dataclasses** |
| `@abstractmethod` | marks a method every subclass must provide | **Interfaces** |

`@dataclass` is applied to a class rather than a function. That works because a class is a value
too, and a decorator is only a function that takes one value and returns another.

### Putting it together: a flaky instrument

An instrument on a bad connection drops its first few requests, and the program reading it should
retry, and log what it does. That uses nearly everything above at once: two decorators written once,
one of them taking a setting, stacked, on a method, with `functools.wraps` keeping the log readable.

The comparison at the start put the same two decorators on plain functions. Here they go on a
method, and then the order of the two lines is reversed to show what that changes. `log_calls` and
`retry` are the ones already defined in this notebook.


In [22]:
class Instrument:
    """A sensor that drops its first few requests, the way a bad connection does."""

    def __init__(self, name, failures, reading):
        self.name = name
        self.failures = failures
        self.reading = reading

    def __repr__(self):
        return f"Instrument({self.name!r})"

    @log_calls
    @retry(times=3)
    def read(self):
        if self.failures > 0:
            self.failures -= 1
            raise ConnectionError("no response")
        return self.reading


print("result:", Instrument("Tromso", failures=2, reading=-4.1).read())


  calling read(Instrument('Tromso'))
  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  read returned -4.1
result: -4.1


One call logged, two failures retried inside it, one result. `log_calls` is on the outside, so it
sees the method called once and returning once. The retrying all happens within that single call.

Now an instrument that never answers.


In [23]:
try:
    Instrument("Bodo", failures=5, reading=-1.9).read()
except ConnectionError as error:
    print("gave up:", error)


  calling read(Instrument('Bodo'))
  attempt 1 of 3 failed: no response
  attempt 2 of 3 failed: no response
  attempt 3 of 3 failed: no response
gave up: read failed 3 times


`calling read` printed and `read returned` did not, because the exception went straight through
`log_calls`'s wrapper on its way out. A wrapper that calls the original does not need to catch its
exceptions in order to let them pass.

Now the same two decorators the other way round, on the `read_sensor` function from the section on
settings.


In [24]:
outcomes = iter(["timeout", "timeout", -4.1])


@retry(times=3)
@log_calls
def read_sensor():
    outcome = next(outcomes)
    if outcome == "timeout":
        raise ConnectionError("no response")
    return outcome


print("result:", read_sensor())


  calling read_sensor()
  attempt 1 of 3 failed: no response
  calling read_sensor()
  attempt 2 of 3 failed: no response
  calling read_sensor()
  read_sensor returned -4.1
result: -4.1


Now `log_calls` is on the inside, so `retry` calls it on every attempt, and every attempt is logged.
Each failed attempt shows a `calling` line and no `returned`, which is what somebody debugging the
connection would want to see. The first order shows the caller one call; this one shows every try.

Neither order is wrong. They answer different questions, and the only difference between them is
which line comes first.

### Where each part came from

| In the instrument | What it relies on | The section that showed it |
|---|---|---|
| `@log_calls` over `@retry(times=3)` | stacked decorators apply from the bottom up | Stacking decorators |
| `retry(times=3)` | a setting means one more layer, a function that builds the decorator | A decorator that takes an argument |
| `wrapper` using both `times` and `func` | a closure remembers the variables around it | Your first wrapper |
| `self` passing through `*args` | a method is a function, and `self` is its first argument | Decorating a method |
| the log saying `read`, not `wrapper` | `functools.wraps` copies the name across | Keeping the name: `functools.wraps` |
| the `ConnectionError` reaching the caller | a wrapper lets exceptions through unless it catches them | the instrument that never answers |
| `Instrument('Tromso')` in the log | a `__repr__` | the **Dunder Methods** notebook |


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/06-decorators-solutions.ipynb).

**1.** Write `@upper`, which makes a function's returned string upper case. Apply it with `@` to one
function, then apply it to a second function by ordinary assignment instead, and show both give the
same output.


In [25]:
# your code here


**2.** Write `@trace`, which prints the function's name and arguments before the call and the
returned value after it. Use `functools.wraps`, apply it to a function taking two arguments, and
print the decorated function's `__name__`.


In [26]:
# your code here


**3.** Write `@count_calls`, which records on the decorated function how many times it has been
called. Call a decorated function three times, then print the count.


In [27]:
# your code here


**4.** Write `@clamp(low, high)`, a decorator that takes settings and forces a function's numeric
return value into that range, so that `@clamp(-90, 60)` turns `999` into `60` and `-120` into `-90`.
Show both.


In [28]:
# your code here


**5.** Stack `@clamp(-90, 60)` and `@trace` on a function that returns `999`, once in each order.
Say in a comment why `trace` reports a different returned value in the two versions.


In [29]:
# your code here


**6.** Rewrite the `TemporaryUnit` context manager from the **Context Managers and Iterators**
notebook as a generator decorated with `@contextlib.contextmanager`. Show the unit restored after a
normal block and after a block that raises.


In [30]:
# your code here


## Common errors

### TypeError: the decorator forgot to return the wrapper

A decorator's last line is `return wrapper`. Without it the decorator returns `None`, and `@` puts
`None` under the function's name.


In [31]:
def no_return(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)


@no_return
def total(values):
    return sum(values)


print("total is now:", total)
total([1.0, 3.0])


total is now: None


TypeError: 'NoneType' object is not callable

The error is on the call, but the mistake is in the decorator. `'NoneType' object is not callable`
on a decorated name almost always means this, and printing the name, as the cell does, shows it
immediately.

### TypeError: parentheses on a decorator that takes none

`log_calls` takes a function and nothing else.


In [32]:
@log_calls()
def total(values):
    return sum(values)


TypeError: log_calls() missing 1 required positional argument: 'func'

`@log_calls()` calls `log_calls` with nothing, before there is a function to give it. This error
arrives when the function is defined, not when it is called.

Parentheses belong after a decorator's name only when it takes settings, as `retry` does.

### TypeError: no parentheses on a decorator that needs them

The reverse mistake is worse, because defining the function raises nothing.


In [33]:
@retry
def read_once():
    return -4.1


print("read_once now holds:", read_once.__qualname__)
read_once()


read_once now holds: retry.<locals>.decorator


TypeError: retry.<locals>.decorator() missing 1 required positional argument: 'func'

`@retry` passed the function to `retry` as `times`. `retry` returned `decorator`, and `decorator`
took over the name `read_once`. Calling it then ran `decorator` with no function to decorate.

`__qualname__` shows what a name really holds, including where it was defined, which makes it the
fastest way to diagnose this.

### TypeError: a message that contradicts your own `def`

A wrapper written without `*args` and `**kwargs` cannot accept the original's arguments. With
`functools.wraps` on it, the resulting message looks wrong.


In [34]:
def strict(func):
    @functools.wraps(func)
    def wrapper():
        return func()
    return wrapper


@strict
def total(values):
    return sum(values)


total([1.0, 3.0])


TypeError: total() takes 0 positional arguments but 1 was given

`def total(values)` plainly takes one argument, and the message says `total()` takes none.

The message is really about `wrapper`, which does take none. `functools.wraps` copied the name
`total` onto it, and the error reports the copied name. Without `wraps`, the same mistake names the
real culprit.


In [35]:
def strict_unwrapped(func):
    def wrapper():
        return func()
    return wrapper


@strict_unwrapped
def total(values):
    return sum(values)


try:
    total([1.0, 3.0])
except TypeError as error:
    print("TypeError:", error)


TypeError: strict_unwrapped.<locals>.wrapper() takes 0 positional arguments but 1 was given


That is not a reason to leave `wraps` out. It is a reason to know that when an error about a
decorated function contradicts its own `def`, the wrapper is the one speaking. The fix is
`def wrapper(*args, **kwargs)`, every time.

### The quiet one: the wrapper forgot to return the result

`wrapper` calls the original, and then does not hand its answer back.


In [36]:
def forgets(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)
    return wrapper


@forgets
def total(values):
    return sum(values)


print("total([1.0, 3.0]) is", total([1.0, 3.0]))


total([1.0, 3.0]) is None


`total` ran and computed `4.0`, and `wrapper` threw it away. Nothing raised. Every call to every
function this decorator touches now returns `None`, and the first error, when it comes, will be
somewhere downstream, such as `round` raising `TypeError` on `None`, far from the decorator that
caused it.

`result = func(*args, **kwargs)` followed by `return result` is the shape to hold on to. A wrapper
that calls the original without returning its value is almost always this bug.


## Recap

- Decorators pay off when one behavior surrounds many functions: written once, attached with
  one line, and changed in one place.
- `@decorator` above `def name` means `name = decorator(name)`, run once when the function is
  defined.
- A decorator is a function that takes a function and returns one, usually an inner `wrapper`.
- `wrapper(*args, **kwargs)` accepts whatever the original accepts and passes all of it along.
- `wrapper` reaches the original through a closure, and must `return` its result.
- `functools.wraps` keeps the original's name and docstring, and stores the original as
  `__wrapped__`.
- A decorator with settings, such as `@retry(times=3)`, is a function that builds the decorator:
  three layers.
- Stacked decorators apply from the bottom up, and the order changes behavior.
- On a method, `self` travels through `*args` like any other argument.
- A decorator can return the function unchanged, after registering it somewhere.
- `functools.cache` remembers results. Use it only when the answer depends on the arguments alone.
- `contextlib.contextmanager` turns a generator into a context manager, and needs `try` and
  `finally` to clean up on an error.
- An error about a decorated function that contradicts its own `def` is coming from the wrapper.


## What is next

The **Properties** notebook, and the first decorator this guide uses rather than writes.
`@property` turns a method into something read like an attribute, so a station can compute its mean
whenever `station.mean` is read, and check a value before storing it, without any caller changing
how they write it.


---

&#8592; **Previous:** [Context Managers and Iterators](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/05-context-managers-and-iterators.ipynb)  &nbsp;·&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)  &nbsp;·&nbsp;  **Next:** [Properties](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/07-properties.ipynb) &#8594;
